
## Color Picker and JSON Updater Notebook

### Overview
This Jupyter Notebook provides an interactive tool to load, modify, and save color data stored in a JSON file. It displays color sets as buttons, allowing users to select and change colors using a color picker. The modified colors can then be saved back to an output JSON file.

### Usage
Run the notebook to load colors from the JSON file. Click a button to select a color, adjust it using the color picker, and save the changes using the "Save Colors" button. The updated colors are written to a new JSON file, preserving the original structure.

The notebook works with JSON files formatted as follows:

```json
{
    "sets": [
        {
            "setName": "Example Set",
            "colors": [
                {"label": "Color 1", "hex": "#ff0000"},
                {"label": "Color 2", "hex": "#00ff00"},
                {"label": "Color 3", "hex": "#0000ff"}
            ]
        }
    ]
}
```

Each set has a name (`setName`) and contains a list of colors, where each color has a label and a hex code.

The paths to the input and output JSON files are defined in the code:

```python
json_path = '/zata/zippy/kresgeb/psych_screen/paper_data_processing/colors/test.json'
output_json_path = '/zata/zippy/kresgeb/psych_screen/paper_data_processing/colors/output.json'
```

By default, the output JSON file will overwrite any existing file at the specified path. If users want to preserve multiple versions, they should modify `output_json_path` before running the code cell.

### Disclaimer
This notebook was AI-generated with little to no assurance of robustness and no consideration for maintainability. Users should verify its functionality before relying on it.
```


In [2]:
import ipywidgets as widgets
from IPython.display import display
import json

# Define the path to the JSON file
json_path = '/zata/zippy/kresgeb/psych_screen/paper_data_processing/colors/k16_like_manual.json'
output_json_path = '/zata/zippy/kresgeb/psych_screen/paper_data_processing/colors/output.json'  # Path to save the updated colors

# Load JSON data
with open(json_path, 'r') as f:
    data = json.load(f)

# Create widgets for each set of colors
color_sets = []

# Keep track of buttons and corresponding color data
color_buttons = []
button_to_color_mapping = {}  # Maps button objects to their color data

for color_set in data['sets']:
    set_name_label = widgets.Label(value=color_set['setName'])  # Label for the set name
    color_buttons_row = []

    set_data = {'setName': color_set['setName'], 'colors': []}

    for color in color_set['colors']:
        color_button = widgets.Button(
            layout=widgets.Layout(width='50px', height='50px'),
            style={'button_color': color['hex']},
            tooltip=color['label']  # 👈 label shows on hover
        )
        color_buttons_row.append(color_button)

        # Map button to its corresponding color data
        color_data = {'label': color['label'], 'hex': color['hex']}
        set_data['colors'].append(color_data)
        button_to_color_mapping[color_button] = color_data  # Track mapping

    # Store set data
    color_buttons.append(set_data)

    # Arrange color buttons horizontally
    color_buttons_row = widgets.HBox(color_buttons_row)
    # Combine set name and color buttons in a VBox (set name on top)
    color_set_widget = widgets.VBox([set_name_label, color_buttons_row])
    color_sets.append(color_set_widget)

# Function to change color of a selected square
color_picker = widgets.ColorPicker(description='Pick a color:')
selected_square = None

def select_square(square):
    def on_click(b):
        global selected_square
        selected_square = square
        color_picker.value = button_to_color_mapping[square]['hex']  # Update color picker
    return on_click

# Assign click handlers to color buttons only
for color_set_widget in color_sets:
    for square in color_set_widget.children[1].children:  # Color buttons are in the HBox (second child)
        square.on_click(select_square(square))

# Function to change the color of the selected square and update mapping
def change_color(change):
    global selected_square
    if selected_square and selected_square in button_to_color_mapping:
        new_color = change['new']
        selected_square.style.button_color = new_color  # Update button color
        button_to_color_mapping[selected_square]['hex'] = new_color  # Update mapping

# Observe color picker changes
color_picker.observe(change_color, names='value')

# Function to save the colors back to a JSON file
def save_colors(_):
    new_data = {'sets': color_buttons}  # color_buttons is already tracking updates

    # Write the updated color data to the output JSON file
    with open(output_json_path, 'w') as f:
        json.dump(new_data, f, indent=4)
    print(f"Colors saved to {output_json_path}")

# Add a button to save the colors to a new JSON file
save_button = widgets.Button(description="Save Colors")
save_button.on_click(save_colors)

# Display the color sets, color picker, and save button
display(widgets.VBox(color_sets), color_picker, save_button)


ColorPicker(value='black', description='Pick a color:')

Button(description='Save Colors', style=ButtonStyle())